# 面试题：Bloom Filter 为什么能防缓存穿透，删除和扩容怎样处理？

Bloom Filter 用固定 bit array 和多个 hash 判断“可能存在/一定不存在”。本 Notebook 手写最优参数、bit packing、double hashing、插入/查询、理论与实测 FPR、Counting Bloom 安全删除、分层扩容、缓存穿透仿真和可信快照。

不调用 bloom 库；重点解释无假阴性的前提、false positive 成本、删除陷阱和版本迁移。

In [ ]:
import copy,hashlib,json,math,warnings
from collections import Counter
from types import MappingProxyType
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")
import numpy as np
RNG76=np.random.default_rng(7601)
def canonical76(x): return json.dumps(x,sort_keys=True,separators=(",",":"))
def sha76(x): return hashlib.sha256(x).hexdigest()
assert RNG76 is not None

## 1. 从容量与目标 FPR 推导 `m/k`

对预计 `n` 个 key、目标 false positive `p`，位数 `m≈-n ln p/(ln2)^2`，hash 数 `k≈m/n ln2`。插入量超过容量会让 FPR 快速恶化，因此 capacity 不是建议值，而是扩容/重建触发条件。

参数向上取整，`k>=1`；编码/hash seed/digest 都要绑定快照。

In [ ]:
def bloom_params76(capacity,target_fpr):
    if capacity<1 or not 0<target_fpr<1: raise ValueError("bloom_parameter_contract")
    m=math.ceil(-capacity*math.log(target_fpr)/(math.log(2)**2)); k=max(1,round(m/capacity*math.log(2))); return m,k
m76,k76=bloom_params76(1000,.01)
assert 9500<m76<9700 and 6<=k76<=7
theoretical76=(1-math.exp(-k76*1000/m76))**k76
assert abs(theoretical76-.01)<.003
try: bloom_params76(0,.1); raise AssertionError("zero capacity accepted")
except ValueError as e: assert str(e)=="bloom_parameter_contract"

## 2. Double hashing 与 bit packing

一次 SHA-256 拆成两个 64-bit 值，用 `h_i=h1+i*h2 mod m` 产生 k 个位置，避免做 k 次加密 hash。若 `h2=0` 强制为奇数常量。bit array 使用 uint8 packing，实际占 `ceil(m/8)` bytes。

Python 内置 hash 不稳定，不能用于持久化 filter；key 编码必须固定。

In [ ]:
def positions76(key,m,k,seed=7601):
    if not isinstance(key,str): raise TypeError("key_string_contract")
    digest=hashlib.sha256(seed.to_bytes(8,"little")+key.encode()).digest(); h1=int.from_bytes(digest[:8],"little"); h2=int.from_bytes(digest[8:16],"little") or 0x9E3779B97F4A7C15
    return tuple((h1+i*h2)%m for i in range(k))
pos_probe76=positions76("user:42",m76,k76)
assert len(pos_probe76)==k76 and all(0<=p<m76 for p in pos_probe76)
assert pos_probe76==positions76("user:42",m76,k76) and pos_probe76!=positions76("user:43",m76,k76)
try: positions76(42,m76,k76); raise AssertionError("non-string key accepted")
except TypeError as e: assert str(e)=="key_string_contract"

## 3. 手写 packed Bloom Filter

add 将所有 bit 置 1；contains 只有在所有 bit 为 1 时返回“可能存在”。因此合法插入后不能出现 false negative，除非 bit array 被损坏、hash/seed 改变或误做删除。

Bloom 不保存 key，也无法枚举成员；只能放在 authoritative store 前做 negative gate。

In [ ]:
class BloomFilter76:
    def __init__(self,capacity,target_fpr,seed=7601):
        self.capacity=capacity; self.target_fpr=target_fpr; self.m,self.k=bloom_params76(capacity,target_fpr); self.seed=seed; self.bits=np.zeros((self.m+7)//8,dtype=np.uint8); self.insertions=0
    def _get(self,p): return bool(self.bits[p//8] & (1<<(p%8)))
    def _set(self,p): self.bits[p//8] |= np.uint8(1<<(p%8))
    def add(self,key):
        for p in positions76(key,self.m,self.k,self.seed): self._set(p)
        self.insertions+=1
    def __contains__(self,key): return all(self._get(p) for p in positions76(key,self.m,self.k,self.seed))
    def estimated_fpr(self): return (1-math.exp(-self.k*self.insertions/self.m))**self.k
bloom76=BloomFilter76(1000,.01); inserted76=[f"known-{i}" for i in range(1000)]
for key in inserted76: bloom76.add(key)
assert bloom76.m==m76 and bloom76.k==k76
assert all(key in bloom76 for key in inserted76) and bloom76.insertions==1000
assert bloom76.bits.nbytes==math.ceil(bloom76.m/8) and bloom76.bits.sum()>0
assert abs(bloom76.estimated_fpr()-theoretical76)<1e-12

## 4. 实测 false positive 与容量超载

对 20,000 个未插入 key 测 FPR，并用二项标准误允许合理波动。再插入额外 1,000 个 key，证明 estimated FPR 显著增加。Bloom 的 false positive 只造成一次多余后端查询，不能被当作存在证明。

测试 key 必须与插入集不重叠；小样本 FPR 抖动很大，不能据此误判 hash 实现。

In [ ]:
negatives76=[f"missing-{i}" for i in range(20000)]; empirical76=np.mean([key in bloom76 for key in negatives76]); se76=math.sqrt(theoretical76*(1-theoretical76)/len(negatives76))
assert abs(empirical76-theoretical76)<4*se76+.002 and 0<empirical76<.03
before_overload76=bloom76.estimated_fpr()
for i in range(1000,2000): bloom76.add(f"known-{i}")
after_overload76=bloom76.estimated_fpr()
assert after_overload76>before_overload76*5 and all(f"known-{i}" in bloom76 for i in range(1000,2000))
assert all(key in bloom76 for key in inserted76)

## 5. Counting Bloom 与安全删除

Counting Bloom 将 bit 换成小整数 counter，add 加一、remove 减一。删除一个从未插入的 key 会因为碰撞把别人的 counter 减掉，制造 false negative；因此 wrapper 维护 authoritative multiplicity，只允许删除确实插入的 key。

counter 需要溢出检查。真实系统通常仍从主存储/变更日志驱动删除，而不是信任外部请求。

In [ ]:
class CountingBloom76:
    def __init__(self,capacity,target_fpr,seed=7602): self.m,self.k=bloom_params76(capacity,target_fpr); self.seed=seed; self.counts=np.zeros(self.m,dtype=np.uint16); self.inventory=Counter()
    def add(self,key):
        pos=positions76(key,self.m,self.k,self.seed)
        if any(self.counts[p]==np.iinfo(self.counts.dtype).max for p in pos): raise OverflowError("counter_overflow")
        for p in pos:self.counts[p]+=1
        self.inventory[key]+=1
    def remove(self,key):
        if self.inventory[key]<=0: raise KeyError("remove_unknown_key")
        for p in positions76(key,self.m,self.k,self.seed):
            if self.counts[p]==0: raise RuntimeError("counter_corruption")
            self.counts[p]-=1
        self.inventory[key]-=1
    def contains(self,key): return all(self.counts[p]>0 for p in positions76(key,self.m,self.k,self.seed))
counting76=CountingBloom76(100,.01); counting76.add("a"); counting76.add("a"); counting76.add("b"); counting76.remove("a")
assert counting76.contains("a") and counting76.contains("b") and counting76.inventory["a"]==1
counting76.remove("a"); assert not counting76.contains("a") and counting76.contains("b")
try: counting76.remove("never"); raise AssertionError("unknown deletion accepted")
except KeyError as e: assert e.args[0]=="remove_unknown_key"

## 6. 分层扩容而非原地改 `m/k`

旧 bit 不能映射到新 `m/k`。Scalable Bloom 在当前层达到 capacity 后追加更大、目标 FPR 更低的新层；查询检查所有层，整体 FPR 上界近似各层 FPR 之和。删除仍需 Counting 或重建。

这里每层容量翻倍、FPR 减半，保留 layer manifest，支持逐层回收。

In [ ]:
class ScalableBloom76:
    def __init__(self,initial_capacity=100,base_fpr=.02):
        if initial_capacity<1: raise ValueError("initial_capacity_contract")
        self.base_capacity=int(initial_capacity); self.base_fpr=base_fpr; self.layers=[BloomFilter76(self.base_capacity,base_fpr,7700)]
    def add(self,key):
        layer=self.layers[-1]
        if layer.insertions>=layer.capacity:
            i=len(self.layers); self.layers.append(BloomFilter76(self.base_capacity*(2**i),self.base_fpr/(2**i),7700+i)); layer=self.layers[-1]
        layer.add(key)
    def contains(self,key): return any(key in layer for layer in self.layers)
scalable76=ScalableBloom76(100,.02)
for i in range(350): scalable76.add(f"x-{i}")
assert len(scalable76.layers)==3 and all(scalable76.contains(f"x-{i}") for i in range(350))
assert [l.capacity for l in scalable76.layers]==[100,200,400]
assert sum(l.target_fpr for l in scalable76.layers)<.04

## 7. 缓存穿透 gate 与失效边界

对 1,000 个真实键和 10,000 个恶意不存在键，Bloom 能挡住绝大多数后端查询；false positives 仍去 authoritative DB 并返回 miss。新合法键必须先/原子地进入 DB 与 filter，否则可能被 filter 当不存在形成假阴性业务错误。

删除后旧 Bloom bit 留着只是额外查询，不会返回错误值；cache value 的 TTL/一致性是另一层问题。

In [ ]:
db76={key:f"value-{i}" for i,key in enumerate(inserted76)}; gate76=BloomFilter76(1000,.01)
for key in db76: gate76.add(key)
backend_queries76=0; returned76=0
for key in list(db76)[:100]+negatives76[:10000]:
    if key not in gate76: continue
    backend_queries76+=1
    if key in db76: returned76+=1
false_backend76=backend_queries76-100
assert returned76==100 and false_backend76<250
assert backend_queries76<.04*(10100) and backend_queries76>=100
assert all(key in gate76 for key in list(db76)[:100])

## 8. 快照、checksum 与面试总结

manifest 绑定 capacity/target FPR、m/k、seed/hash/key encoding、insertions、dtype 和 bit bytes。loader 从实际 bit array 重算 digest，并拒绝配置或内容篡改。快照与 DB commit point 必须一致。

面试回答顺序：negative gate 语义 → m/k/FPR → bit/hash → 无假阴性前提 → Counting 删除 → scalable layers → DB 一致性和监控。

In [ ]:
def bits_digest76(a):
    v=np.ascontiguousarray(a); return sha76(str(v.dtype).encode()+canonical76(list(v.shape)).encode()+v.tobytes())
manifest76={"artifact_id":"bloom-cache-gate-v1","capacity":gate76.capacity,"target_fpr":gate76.target_fpr,"m":gate76.m,"k":gate76.k,"seed":gate76.seed,"hash":"sha256-double-v1","encoding":"utf8","insertions":gate76.insertions,"bits_digest":bits_digest76(gate76.bits),"db_snapshot":sha76(canonical76(sorted(db76)).encode())}
TRUST76=MappingProxyType({manifest76["artifact_id"]:sha76(canonical76(manifest76).encode())})
def load_bloom76(m,bloom):
    actual=copy.deepcopy(m); actual["bits_digest"]=bits_digest76(bloom.bits); actual["insertions"]=bloom.insertions
    if TRUST76.get(actual.get("artifact_id"))!=sha76(canonical76(actual).encode()): raise RuntimeError("untrusted_bloom_snapshot")
    if (actual["m"],actual["k"],actual["seed"])!=(bloom.m,bloom.k,bloom.seed): raise RuntimeError("bloom_protocol")
    return bloom
loaded76=load_bloom76(manifest76,gate76)
assert "known-1" in loaded76 and isinstance(TRUST76,MappingProxyType)
forged76=copy.deepcopy(gate76); forged76.bits[0]^=1
try: load_bloom76(manifest76,forged76); raise AssertionError("forged bloom accepted")
except RuntimeError as e: assert str(e)=="untrusted_bloom_snapshot"
print({"m":m76,"k":k76,"theoretical_fpr":round(theoretical76,4),"empirical_fpr":round(float(empirical76),4),"backend_queries":backend_queries76})

## 9. 复杂度、失败模式与来源

add/query 为 `O(k)`，内存 `m` bits；Counting 变为 `m * counter_bits`。常见错误：把 maybe 当 true、原地扩 m/k、直接清 bit 删除、用 Python hash、容量超载不监控、新 DB key 未进 filter、快照与 DB 不一致和 FPR 测试集与插入集重叠。

- Bloom, [Space/Time Trade-offs in Hash Coding with Allowable Errors](https://dl.acm.org/doi/10.1145/362686.362692), 1970。
- Broder & Mitzenmacher, [Network Applications of Bloom Filters](https://www.eecs.harvard.edu/~michaelm/postscripts/im2005b.pdf)。
- Almeida et al., [Scalable Bloom Filters](https://gsd.di.uminho.pt/members/cbm/ps/dbloom.pdf)。